In [2]:
import os
import urllib.request

if not os.path.exists("the-verdixt.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt")
    file_path = "the-verdixt.txt"
    urllib.request.urlretrieve(url, file_path)


In [3]:
with open("the-verdixt.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [4]:
raw_text

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [5]:
len(raw_text)

20479

In [6]:
import re

text = "Hello World. This is a test."
result = re.split(r'(\s)', text)

result

['Hello', ' ', 'World.', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test.']

In [7]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ' ', 'World', '.', '', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [8]:
result = [item for item in result if item.strip()]
print(result)

['Hello', 'World', '.', 'This', 'is', 'a', 'test', '.']


In [9]:
result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
result = [item for item in result if item.strip()]
print(result)
preprocessed = result
print(len(result))

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter', '--', 'deploring', 'his', 'unaccountable', 'abdication', '.', '"', 'Of', 'course', 'it', "'", 's', 'going', 'to', 'send', 'the', 'value', 'of', 'my', 'picture', "'", 'way', 'up', ';', 'but', 'I', 'don', "'", 't', 'think', 'of', 'that', ',

In [10]:
preprocessed[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

In [11]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [12]:
vocab = {token:integer for integer,token in enumerate(all_words)}

In [13]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [14]:
tokenizer = SimpleTokenizerV1(vocab)

In [15]:
text = """"It's the last he painted, you know,"
        Mrs. Gisburn said with pardonable pride."""

In [16]:
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [17]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [18]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [19]:
len(vocab)

1132

In [20]:
for s,i in enumerate(list(vocab.items())[-5:]):
    print(i)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [21]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        #can handle words not in dictionary
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [22]:
tokenizer = SimpleTokenizerV2(vocab)

In [23]:
text = "heleo worlds"
print(tokenizer.encode(text))

[1131, 1131]


In [24]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|> <|unk|>


##### BYTE PAIR ENCODING
##### Tries to break longer unknown words into smaller known ones for ex. someunknownword


In [25]:
import tiktoken

In [26]:
tiktoken.__version__

'0.13.0'

In [27]:
tokenizer = tiktoken.get_encoding("gpt2")

In [28]:
tokenizer.encode("Hello world")

[15496, 995]

In [29]:
tokenizer.decode([15496, 995])

'Hello world'

##### Data Sampling with Sliding Window

In [30]:
with open("the-verdixt.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [31]:
enc_sample = enc_text[50:]

In [32]:
len(enc_sample)

5095

In [33]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287]


In [34]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    
    print(context, "----> ", desired) 
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))
    print()

[290] ---->  4920
 and ---->  established

[290, 4920] ---->  2241
 and established ---->  himself

[290, 4920, 2241] ---->  287
 and established himself ---->  in

[290, 4920, 2241, 287] ---->  257
 and established himself in ---->  a



##### For loading datasets and stuff

In [35]:
import torch

In [36]:
torch.__version__

'2.12.0+cu130'

In [37]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        #Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        #using a sliding window to chunk the book into overlapping sequence of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i+max_length]
            target_chunk = token_ids[i+1 : i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [38]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    
    #inititalize tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    #create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    #create dataloader
    dataloader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle, 
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [39]:
with open("the-verdixt.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [40]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [41]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[1807, 3619,  402,  271]]), tensor([[ 3619,   402,   271, 10899]])]


In [42]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("Targets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### Creating Token Embeddings
Instead of representing a token with a single number, we represent it as a vector of numbers. These vectors are called as Token Embeddings

In [43]:
input_ids = torch.tensor([2, 3, 5, 1])

In [44]:
tokenizer.n_vocab

50257

* An embedding layer is a learnable lookup table that converts token IDs into dense vectors (embeddings).
* Embeddings start random

In [45]:
vocab_size = 6
output_dim = 3


torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(tokenizer.n_vocab, output_dim)

In [46]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035],
        [-0.5880,  0.3486,  0.6603],
        [-0.2196, -0.3792,  0.7671],
        ...,
        [-0.5931,  1.0895, -0.6854],
        [ 0.7447,  0.5803, -0.4246],
        [-0.3130,  0.7558, -1.2656]], requires_grad=True)


In [47]:
embedding_layer(torch.tensor([3]))

tensor([[-1.1925,  0.6984, -1.4097]], grad_fn=<EmbeddingBackward0>)

In [48]:
embedding_layer(input_ids)

tensor([[-0.2196, -0.3792,  0.7671],
        [-1.1925,  0.6984, -1.4097],
        [ 0.2692, -0.0770, -1.0205],
        [-0.5880,  0.3486,  0.6603]], grad_fn=<EmbeddingBackward0>)

### Encoding Word Positions

In [49]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

this creates a 2D Matrix of 50257 rows and 256 columns

In [50]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False
    )
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [51]:
print("Token IDs:\n", inputs)
print("Target IDs:\n", targets)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Target IDs:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])

Inputs shape:
 torch.Size([8, 4])


In [52]:
token_embeddings = token_embedding_layer(inputs)
token_embeddings.shape

torch.Size([8, 4, 256])

* You have 1 batch = 8 sequence
* One sequence = 4 tokens
* Now each token is converted to a random 256 sized vector

In [53]:
token_embeddings[0, 0]

tensor([ 4.5925e-01,  7.6589e-01, -1.7269e+00,  2.7519e-01,  2.3155e-01,
        -4.7319e-01, -3.9847e-01,  1.7265e+00,  8.9715e-01, -1.0197e-01,
         5.1204e-01, -1.6805e+00, -8.0522e-01,  1.1375e+00, -9.9466e-01,
        -9.5338e-01,  8.3238e-01,  1.3210e+00,  9.5062e-01,  1.2799e+00,
         1.1932e+00,  6.7199e-01, -6.2126e-01,  5.5973e-01, -6.0924e-01,
        -9.0394e-02, -1.4039e+00, -4.1480e-01,  2.5625e-01, -9.7720e-03,
        -2.2002e+00, -9.0485e-01, -4.2069e-01, -4.4801e-01,  9.3138e-01,
         2.4970e+00, -7.5886e-01, -2.9304e-03,  4.6033e-01, -1.2624e+00,
         3.7764e-01,  8.1343e-01, -1.2823e+00, -1.2091e-01, -3.3686e-01,
         1.6070e+00, -5.9332e-01, -1.6217e+00,  1.2313e+00, -1.8392e+00,
        -1.2068e+00, -2.5405e+00,  1.9803e+00, -1.2905e+00,  1.7485e+00,
         8.1073e-01,  1.1342e+00, -1.7977e+00, -1.0314e+00, -1.2450e+00,
        -8.8608e-01,  7.4233e-02,  1.0778e+00, -4.8303e-01, -5.6789e-01,
         7.1249e-01,  8.5286e-01,  7.1491e-01, -1.0

##### What we are doing now is trying to get the positional information as well

##### Positonal Embeddings are added to the Token Embeddings vectors to create the input for the LLM. Both have the same dimension
##### In reality the positional embeddings also start with random numbers, they also get optimized later during the Training Phase.

In [54]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [55]:
print(pos_embedding_layer.weight)
# Randoms

Parameter containing:
tensor([[-1.3798,  1.3476, -0.3612,  ..., -0.7712,  0.1523, -0.5973],
        [ 0.3611, -0.5228, -0.2888,  ...,  0.8571,  0.2221,  0.1976],
        [ 1.2194,  0.8234,  0.2277,  ...,  2.5752, -1.7081, -0.5515],
        [-0.5765, -1.6450, -1.3456,  ...,  0.7075,  0.0123, -1.2205]],
       requires_grad=True)


In [56]:
torch.arange(max_length)

tensor([0, 1, 2, 3])

In [57]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [58]:
token_embeddings.shape

torch.Size([8, 4, 256])

In [60]:
pos_embeddings.shape

torch.Size([4, 256])

* The position embeddings would be all added to each batch separately
* token_embeddings[0] + pos_embeddings
* token_embeddings + pos_embeddings -> First of this output and the above one would be same

In [62]:
token_embeddings[0] + pos_embeddings

tensor([[-0.9206,  2.1135, -2.0881,  ..., -2.4569, -0.8164, -1.1871],
        [ 0.3567,  1.1969,  0.0850,  ...,  0.3700,  0.8277,  0.8440],
        [ 1.1740, -0.0410, -1.7390,  ...,  3.6197, -4.8771,  1.4966],
        [-0.9454, -1.5052, -1.5424,  ...,  0.6199,  1.6592, -1.4935]],
       grad_fn=<AddBackward0>)

In [64]:
token_embeddings + pos_embeddings

tensor([[[-0.9206,  2.1135, -2.0881,  ..., -2.4569, -0.8164, -1.1871],
         [ 0.3567,  1.1969,  0.0850,  ...,  0.3700,  0.8277,  0.8440],
         [ 1.1740, -0.0410, -1.7390,  ...,  3.6197, -4.8771,  1.4966],
         [-0.9454, -1.5052, -1.5424,  ...,  0.6199,  1.6592, -1.4935]],

        [[-2.4155,  1.5640,  1.5653,  ..., -0.9509, -1.3729, -1.4296],
         [ 1.2617, -0.9205,  2.0868,  ...,  1.2578,  0.1744,  0.0704],
         [ 0.6692,  1.6668,  1.1340,  ...,  2.9328, -2.8202,  0.5882],
         [-0.8770, -2.6726,  0.2737,  ..., -0.3932, -0.8468, -1.2137]],

        [[-1.5554,  1.5362, -0.9213,  ...,  0.5317,  0.0264, -0.3517],
         [-0.7942, -1.0510, -1.2759,  ...,  1.3616,  0.5881,  1.5646],
         [ 0.3293,  1.4538,  0.2219,  ...,  2.8004, -1.3926, -0.5830],
         [-1.1202, -0.0275, -0.2594,  ...,  0.8665, -1.1160,  0.2821]],

        ...,

        [[-0.5048,  1.3744,  0.3595,  ...,  0.5127, -0.4135, -0.8774],
         [ 0.2300, -0.9263, -0.9182,  ...,  1.3744,  0.01

In [63]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
